-----------------------------------------

## **2. Data Modelling**

-----------------------------------------

### Table of Contents </font> <a class="anchor" id='toc'></a> 

* [2.1 Importing the Dataset and Libraries](#p2_1)
* [2.2 Dataset Preparation](#p2_2)
* [2.3 Clustering](#p2_3)
* [2.4 UMAP Plot](#p2_4)
* [2.5 Exporting the Datasets](#p2_5)

# Issues/Questions? 

In March, the 02:00 hour doesn’t exist.
In November, the 01:00 hour happens twice and the two get merged.

with DST did we check these things? 

Should we assume idle hours happens after charging hours? or should we normalize them both when we are doing the hourly? 

Should we keep users with only one session? 

Assumption:
- Charging starts at start.
- Active charging continues for charging_hours.
- The vehicle remains connected until end.
- Energy is delivered at a constant average rate during active charging.

## <font color='#FFFFF'>2.1. Importing the Dataset and Libraries <font><a class="anchor" id="p2_1"></a>
  
[Back to TOC](#toc)

In [162]:
# Data manipulation and analysis
#import umap
import pandas as pd

# Feature scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

import warnings
warnings.filterwarnings('ignore')

In [163]:
%load_ext autoreload
%autoreload 2
#from Functions import *
from Visualizations import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [164]:
ev = pd.read_pickle('Data/ev_clean.pkl')
display(ev.tail(3))

,station,mac_address,start,start_tz,end,end_tz,transaction_date,total_duration,charging_time,energy_kwh,...,lat,lon,fee,ended_by,event_id,driver_postal_code,user_id,total_hours,charging_hours,idle_hours
247858,PALO ALTO CA / BRYANT #4,0024:B100:0002:6D37,2020-02-29 20:10:00,PST,2020-03-01 00:00:02,PST,2020-03-01 00:01:00,0 days 03:50:02,0 days 03:49:47,12.689,...,37.446785,-122.162369,2.92,Plug Out at Vehicle,3234,94568,1554761,3.833889,3.829722,0.004167
247859,PALO ALTO CA / WEBSTER #1,0024:B100:0002:4233,2020-02-29 20:14:00,PST,2020-03-01 00:29:20,PST,2020-03-01 00:30:00,0 days 04:15:20,0 days 04:15:04,25.711,...,37.448284,-122.158272,5.91,Plug Out at Vehicle,6482,94089,3869081,4.255556,4.251111,0.004444
247860,PALO ALTO CA / BRYANT #4,0024:B100:0002:6D37,2020-02-29 21:47:00,PST,2020-02-29 23:18:39,PST,2020-02-29 23:20:00,0 days 01:31:39,0 days 01:31:27,9.245,...,37.446785,-122.162369,2.13,Plug Out at Vehicle,3235,95070,319921,1.527500,1.524167,0.003333


## <font color='#FFFFF'>2.2. Dataset Preparation <font><a class="anchor" id="p2_2"></a>
  
[Back to TOC](#toc)

Our dataset has users without id, including these observations in the user-level clustering would introduce observations that cannot be assigned to a meaningful user. Therefore we decided to remove these observations

In [168]:
u0 = ev[ev["user_id"] == "0"]
print("\nID '0' exists:", len(u0) > 0)
print("\nSessions:", len(ev), "| users:", ev["user_id"].nunique())
print("\nMost active IDs:")
print(ev["user_id"].value_counts().head(8))
if len(u0):
    print("User 0: ", "sessions:", len(u0), "| stations:", u0["station"].nunique(),
          "| distinct driver postal codes:", u0["driver_postal_code"].nunique(),
          "| period:", u0["start"].min().date(), "->", u0["start"].max().date())


ID '0' exists: True

Sessions: 238583 | users: 20585

Most active IDs:
user_id
283441    1103
453469    1051
128279     986
196691     974
2670       881
524259     880
523487     876
804075     792
Name: count, dtype: int64
User 0:  sessions: 447 | stations: 14 | distinct driver postal codes: 0 | period: 2011-08-02 -> 2015-09-02


ID 0 has no driver postal code and appears only in 2011–2015, so it is treated as a default account and excluded

In [ ]:
# Remove missing IDs before converting to string
ev = ev[ev["user_id"].notna()].copy()

ev["user_id"] = ev["user_id"].astype(str).str.strip()

# Exclude default account and empty IDs
ev = ev[
    (ev["user_id"] != "0") &
    (ev["user_id"] != "") &
    (ev["user_id"].str.lower() != "nan")
].copy()

print("Sessions:", len(ev))
print("Users:", ev["user_id"].nunique())

**Hourly conversion: assumptions.** Charging starts at plug-in and runs for `charging_hours` at a constant average power. The car then stays connected, idle, until unplugging. The exact charging profile is not recorded, so this is the only ordering the data supports and it keeps the implied power physically plausible. Occupation uses the original start and end. Timestamps are naive local time, so the two daylight-saving changes per year (a missing hour in March, a repeated hour in November) are ignored. Only hour-of-day shares are used here, so the effect is negligible.

In [ ]:
src = ev[["user_id", "station", "start", "end", "energy_kwh", "charging_hours"]].copy()

# estimated end of active charging, capped at unplugging time
charging_end = src["start"] + pd.to_timedelta(src["charging_hours"], unit="h")
src["charging_end"] = charging_end.where(charging_end < src["end"], src["end"])

# one row per clock hour touched by the session
src["hour_start"] = [
    pd.date_range(s.floor("h"), (e - pd.Timedelta(nanoseconds=1)).floor("h"), freq="h")
    for s, e in zip(src["start"], src["end"])
]
hourly = src.explode("hour_start").dropna(subset=["hour_start"]).copy()
hourly["hour_start"] = pd.to_datetime(hourly["hour_start"])
hourly["hour_end"] = hourly["hour_start"] + pd.Timedelta(hours=1)
hourly["hour"] = hourly["hour_start"].dt.hour

def overlap_minutes(a_start, a_end, b_start, b_end):
    return ((np.minimum(a_end, b_end) - np.maximum(a_start, b_start))
            .dt.total_seconds() / 60).clip(lower=0)

hourly["occupation_minutes"] = overlap_minutes(hourly["start"], hourly["end"],
                                               hourly["hour_start"], hourly["hour_end"])
hourly["charging_minutes"] = overlap_minutes(hourly["start"], hourly["charging_end"],
                                             hourly["hour_start"], hourly["hour_end"])
hourly["idle_minutes"] = (hourly["occupation_minutes"] - hourly["charging_minutes"]).clip(lower=0)

# energy of the hour = average power of the session x charging time in that hour
avg_power = hourly["energy_kwh"] / hourly["charging_hours"].where(hourly["charging_hours"] > 0)
hourly["energy_kwh_h"] = (avg_power * hourly["charging_minutes"] / 60).fillna(0)

hourly["period"] = pd.cut(hourly["hour"], [-1, 5, 11, 17, 23],
                          labels=["night", "morning", "afternoon", "evening"])

energy kept   : 1.0
occupation kept: 1.0
rows: 833875


,user_id,station,start,end,energy_kwh,charging_hours,charging_end,hour_start,hour_end,hour,occupation_minutes,charging_minutes,idle_minutes,energy_kwh_h,period
0,3284,PALO ALTO CA / HAMILTON #1,2011-07-29 20:17:00,2011-07-29 23:20:32,6.249457,1.900833,2011-07-29 22:11:02.999999998,2011-07-29 20:00:00,2011-07-29 21:00:00,20,43.0,43.00,0.00,2.356218,evening
0,3284,PALO ALTO CA / HAMILTON #1,2011-07-29 20:17:00,2011-07-29 23:20:32,6.249457,1.900833,2011-07-29 22:11:02.999999998,2011-07-29 21:00:00,2011-07-29 22:00:00,21,60.0,60.00,0.00,3.287746,evening
0,3284,PALO ALTO CA / HAMILTON #1,2011-07-29 20:17:00,2011-07-29 23:20:32,6.249457,1.900833,2011-07-29 22:11:02.999999998,2011-07-29 22:00:00,2011-07-29 23:00:00,22,60.0,11.05,48.95,0.605493,evening


**Hourly profiles per user.** For each user we sum energy and occupation by hour of day and normalise to shares, so the profile describes *when* someone charges, independent of how much.

In [172]:
user_hourly = (hourly.groupby(["user_id", "hour"])
                     .agg(energy=("energy_kwh_h", "sum"),
                          occupation=("occupation_minutes", "sum"))
                     .reset_index())

def profile(col, prefix):
    p = (user_hourly.pivot(index="user_id", columns="hour", values=col)
                    .reindex(columns=range(24)).fillna(0))          # fill hours with no activity
    p = p.div(p.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)   # shares sum to 1
    p.columns = [f"{prefix}_{h:02d}" for h in range(24)]
    return p

energy_share = profile("energy", "energy_share")
occupation_share = profile("occupation", "occupation_share")

# compact version: share of energy in four periods of the day
period_energy = hourly.pivot_table(index="user_id", columns="period", values="energy_kwh_h",
                                   aggfunc="sum", fill_value=0, observed=True)
period_share = (period_energy.div(period_energy.sum(axis=1).replace(0, np.nan), axis=0)
                             .fillna(0).add_prefix("share_"))
period_share.head(3)

period,share_night,share_morning,share_afternoon,share_evening
user_id,,,,
0,0.051225,0.239531,0.321288,0.387955
1000031,0.000000,0.000000,0.000000,1.000000
1000311,0.000000,0.000000,0.000000,1.000000


In [184]:
# Identify weekday/weekend for each hourly record
# Use hour_start because a charging session can cross midnight

hourly["day_type"] = np.where(
    hourly["hour_start"].dt.dayofweek >= 5,
    "weekend",
    "weekday"
)

# Aggregate energy by user, day type and period
weekday_weekend_energy = (
    hourly.groupby(
        ["user_id", "day_type", "period"],
        observed=True
    )["energy_kwh_h"]
    .sum()
    .unstack(["day_type", "period"], fill_value=0)
)

# Ensure all combinations exist
expected_columns = pd.MultiIndex.from_product(
    [
        ["weekday", "weekend"],
        ["night", "morning", "afternoon", "evening"]
    ],
    names=["day_type", "period"]
)

weekday_weekend_energy = weekday_weekend_energy.reindex(
    columns=expected_columns,
    fill_value=0
)

# Calculate energy shares separately for weekdays and weekends
weekday_weekend_share = weekday_weekend_energy.copy()

for day_type in ["weekday", "weekend"]:

    cols = day_type

    day_total = (
        weekday_weekend_energy[cols]
        .sum(axis=1)
        .replace(0, np.nan)
    )

    weekday_weekend_share[cols] = (
        weekday_weekend_energy[cols]
        .div(day_total, axis=0)
        .fillna(0)
    )

# Flatten column names
weekday_weekend_share.columns = [
    f"share_{day_type}_{period}"
    for day_type, period in weekday_weekend_share.columns
]

weekday_weekend_share.tail()

,share_weekday_night,share_weekday_morning,share_weekday_afternoon,share_weekday_evening,share_weekend_night,share_weekend_morning,share_weekend_afternoon,share_weekend_evening
user_id,,,,,,,,
998031,0.0,0.39262,0.539665,0.067715,0.0,0.0,0.0,0.0
99823,0.0,0.00000,0.284754,0.715246,0.0,1.0,0.0,0.0
998481,0.0,1.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
998781,0.0,0.00000,1.000000,0.000000,0.0,0.0,0.0,0.0
99943,0.0,0.00000,0.551193,0.448807,0.0,0.0,0.0,0.0


**General features per user.** Features are chosen to answer four questions: how much (sessions, energy, power), when (weekend, time of day), whether they hoard (idle ratio) and where (stations, loyalty). Redundant variables are left out on purpose: `charging_ratio` equals 1 − `idle_ratio`, total time equals charging plus idle time, and `total_energy` is roughly sessions × mean energy. Fee is excluded because free charging ended in 2017 (a pricing change, not behaviour), and port and plug type are identical and almost constant (98% Level 2).

In [ ]:
ev["is_weekend"] = (ev["start"].dt.dayofweek >= 5).astype(int)
ev["idle_pos"] = ev["idle_hours"].clip(lower=0)

g = ev.groupby("user_id")
user_general = pd.DataFrame({
    "total_sessions": g.size(),
    "mean_energy": g["energy_kwh"].mean(),
    "idle_ratio": (
        g["idle_pos"].sum() /
        g["total_hours"].sum().replace(0, np.nan)
    ),
    "weekend_ratio": g["is_weekend"].mean(),
    "distinct_stations": g["station"].nunique(),
})

# duration-weighted average power over sessions with charging time
valid = ev[ev["charging_hours"] > 0].groupby("user_id")
user_general["mean_power"] = valid["energy_kwh"].sum() / valid["charging_hours"].sum()

# share of sessions at the user's most-used station
counts = ev.groupby(["user_id", "station"], observed=True).size()
user_general["station_loyalty"] = counts.groupby(level="user_id").max() / user_general["total_sessions"]

# only for profiling clusters afterwards, never used as a feature
user_general["first_year"] = g["start"].min().dt.year
user_general.describe().round(2)

,total_sessions,mean_energy,idle_ratio,weekend_ratio,distinct_stations,mean_power,station_loyalty,first_year
count,20585.00,20585.00,20585.00,20585.00,20585.00,20585.00,20585.00,20585.00
mean,11.59,7.37,0.10,0.35,2.64,4.56,0.72,2016.84
std,44.28,5.14,0.16,0.40,2.75,1.35,0.29,2.07
min,1.00,0.10,0.00,0.00,1.00,0.06,0.10,2011.00
25%,1.00,4.02,0.00,0.00,1.00,3.22,0.50,2016.00
50%,2.00,6.33,0.01,0.14,2.00,5.04,0.82,2017.00
75%,5.00,9.44,0.14,0.67,3.00,5.85,1.00,2019.00
max,1103.00,71.18,0.98,1.00,29.00,8.85,1.00,2020.00


In [182]:
# Charging frequency features per user

# Sort sessions chronologically
ev = ev.sort_values(["user_id", "start"]).copy()

# Calculate days between consecutive sessions
ev["days_since_previous"] = (
    ev.groupby("user_id")["start"]
      .diff()
      .dt.total_seconds() / 86400
)

# Aggregate frequency information
user_frequency = ev.groupby("user_id").agg(
    median_days_between_sessions=("days_since_previous", "median"),
    active_months=("start", lambda x: x.dt.to_period("M").nunique()),
    active_days=("start", lambda x: x.dt.date.nunique())
)

user_frequency.head()

,median_days_between_sessions,active_months,active_days
user_id,,,
1000031,NaN,1,1
1000311,NaN,1,1
100061V,56.993403,2,3
10013,820.044444,3,3
100161V,NaN,1,1


In [187]:
# User profiling variables (not used for clustering)

user_profile = pd.DataFrame(index=user_general.index)

# Activity period
user_profile["first_year"] = g["start"].min().dt.year
user_profile["last_year"] = g["start"].max().dt.year

# Activity regularity
user_profile["active_months"] = user_frequency["active_months"]
user_profile["active_days"] = user_frequency["active_days"]

# Overall consumption
user_profile["total_energy"] = g["energy_kwh"].sum()
user_profile["total_hours"] = g["total_hours"].sum()

# Geographic information
user_profile["driver_postal_code"] = (
    g["driver_postal_code"].agg(
        lambda x: x.dropna().mode().iloc[0]
        if not x.dropna().mode().empty else np.nan
    )
)

# Average charging station coordinates
user_profile["avg_station_lat"] = g["lat"].mean()
user_profile["avg_station_lon"] = g["lon"].mean()

user_profile.head()

,first_year,last_year,active_months,active_days,total_energy,total_hours,driver_postal_code,avg_station_lat,avg_station_lon
user_id,,,,,,,,,
0,2011,2015,NaN,NaN,3084.095225,1257.715833,<NA>,37.444873,-122.159906
1000031,2019,2019,1.0,1.0,4.959000,3.612778,94041,37.444489,-122.160370
1000311,2019,2019,1.0,1.0,13.430000,2.411667,94070,37.444706,-122.160324
100061V,2019,2020,2.0,3.0,34.569000,9.332500,<NA>,37.447722,-122.159546
10013,2015,2019,3.0,3.0,11.177250,2.626389,94061,37.431266,-122.132795


In [188]:
# Select weekday/weekend temporal features
# Drop night shares to avoid perfect linear dependence

temporal_features = weekday_weekend_share.drop(
    columns=[
        "share_weekday_night",
        "share_weekend_night"
    ]
)

# Assemble final user-level feature matrix
users = (
    user_general
    .join(user_frequency[["median_days_between_sessions"]])
    .join(temporal_features)
)

print("Users:", users.shape[0])
print("Features:", users.shape[1])

display(users.head())

Users: 20585
Features: 15


,total_sessions,mean_energy,idle_ratio,weekend_ratio,distinct_stations,mean_power,station_loyalty,first_year,median_days_between_sessions,share_weekday_morning,share_weekday_afternoon,share_weekday_evening,share_weekend_morning,share_weekend_afternoon,share_weekend_evening
user_id,,,,,,,,,,,,,,,
0,447,6.899542,0.273696,0.199105,14,3.376188,0.212528,2011,NaN,0.263173,0.312108,0.381826,0.138065,0.360687,0.414261
1000031,1,4.959000,0.511994,0.000000,1,2.812730,1.000000,2019,NaN,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1000311,1,13.430000,0.002995,0.000000,1,5.585490,1.000000,2019,NaN,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
100061V,3,11.523000,0.368188,0.000000,2,5.862741,0.666667,2019,56.993403,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
10013,3,3.725750,0.008461,0.000000,3,4.292064,0.333333,2015,820.044444,0.000000,0.355298,0.644702,0.000000,0.000000,0.000000


In [189]:
# Define features to include in clustering

columns_to_keep = [
    # Charging intensity
    "total_sessions",
    "mean_energy",

    # Charging frequency
    "median_days_between_sessions",

    # Charging behaviour
    "idle_ratio",
    "weekend_ratio",

    # Charging characteristics
    "mean_power",

    # Station behaviour
    "distinct_stations",
    "station_loyalty",

    # Weekday temporal behaviour
    "share_weekday_morning",
    "share_weekday_afternoon",
    "share_weekday_evening",

    # Weekend temporal behaviour
    "share_weekend_morning",
    "share_weekend_afternoon",
    "share_weekend_evening"
]

# Select clustering features
X = users[columns_to_keep].copy()

print("Clustering dataset:", X.shape)
display(X.head())

Clustering dataset: (20585, 14)


,total_sessions,mean_energy,median_days_between_sessions,idle_ratio,weekend_ratio,mean_power,distinct_stations,station_loyalty,share_weekday_morning,share_weekday_afternoon,share_weekday_evening,share_weekend_morning,share_weekend_afternoon,share_weekend_evening
user_id,,,,,,,,,,,,,,
0,447,6.899542,NaN,0.273696,0.199105,3.376188,14,0.212528,0.263173,0.312108,0.381826,0.138065,0.360687,0.414261
1000031,1,4.959000,NaN,0.511994,0.000000,2.812730,1,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1000311,1,13.430000,NaN,0.002995,0.000000,5.585490,1,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
100061V,3,11.523000,56.993403,0.368188,0.000000,5.862741,2,0.666667,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
10013,3,3.725750,820.044444,0.008461,0.000000,4.292064,3,0.333333,0.000000,0.355298,0.644702,0.000000,0.000000,0.000000


In [192]:
# Validate final user-level dataset

print("Dataset shape:", users.shape)
print("Unique users:", users.index.nunique())

print("\nMissing values:")
print(users.isna().sum())

print("\nDuplicate users:", users.index.duplicated().sum())

print("\nDescriptive statistics:")
display(users.describe().round(3))

# Check temporal shares
weekday_cols = [
    "share_weekday_morning",
    "share_weekday_afternoon",
    "share_weekday_evening"
]

weekend_cols = [
    "share_weekend_morning",
    "share_weekend_afternoon",
    "share_weekend_evening"
]

print("\nWeekday shares:")
display(users[weekday_cols].describe())

print("\nWeekend shares:")
display(users[weekend_cols].describe())

Dataset shape: (20585, 15)
Unique users: 20585

Missing values:
total_sessions                     0
mean_energy                        0
idle_ratio                         0
weekend_ratio                      0
distinct_stations                  0
mean_power                         0
station_loyalty                    0
first_year                         0
median_days_between_sessions    9078
share_weekday_morning              0
share_weekday_afternoon            0
share_weekday_evening              0
share_weekend_morning              0
share_weekend_afternoon            0
share_weekend_evening              0
dtype: int64

Duplicate users: 0

Descriptive statistics:


,total_sessions,mean_energy,idle_ratio,weekend_ratio,distinct_stations,mean_power,station_loyalty,first_year,median_days_between_sessions,share_weekday_morning,share_weekday_afternoon,share_weekday_evening,share_weekend_morning,share_weekend_afternoon,share_weekend_evening
count,20585.000,20585.000,20585.000,20585.000,20585.000,20585.000,20585.000,20585.000,11507.000,20585.000,20585.000,20585.000,20585.000,20585.000,20585.000
mean,11.590,7.373,0.100,0.349,2.644,4.557,0.724,2016.838,132.245,0.119,0.328,0.331,0.090,0.291,0.154
std,44.283,5.136,0.162,0.404,2.755,1.348,0.293,2.065,235.076,0.252,0.378,0.408,0.231,0.395,0.310
min,1.000,0.100,0.000,0.000,1.000,0.055,0.105,2011.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,1.000,4.022,0.004,0.000,1.000,3.221,0.500,2016.000,9.925,0.000,0.000,0.000,0.000,0.000,0.000
50%,2.000,6.332,0.013,0.143,2.000,5.037,0.818,2017.000,42.611,0.000,0.140,0.020,0.000,0.000,0.000
75%,5.000,9.443,0.136,0.667,3.000,5.847,1.000,2019.000,146.938,0.082,0.640,0.745,0.000,0.620,0.084
max,1103.000,71.185,0.981,1.000,29.000,8.850,1.000,2020.000,2672.174,1.000,1.000,1.000,1.000,1.000,1.000



Weekday shares:


,share_weekday_morning,share_weekday_afternoon,share_weekday_evening
count,20585.000000,20585.000000,20585.000000
mean,0.119221,0.327922,0.330983
std,0.252461,0.378357,0.407878
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.139935,0.020059
75%,0.081881,0.639663,0.744567
max,1.000000,1.000000,1.000000



Weekend shares:


,share_weekend_morning,share_weekend_afternoon,share_weekend_evening
count,20585.000000,20585.000000,20585.000000
mean,0.089612,0.291255,0.154396
std,0.230731,0.395293,0.310082
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.620087,0.084477
max,1.000000,1.000000,1.000000


Before proceeding with the code implementation, the definitions of the three scalers are as follows:

<b>Standard Scaler</b> standardizes features by removing the mean and scaling to unit variance. It centers the feature values around 0 and scales them to have a standard deviation of 1, ensuring all features are on a similar scale.

<b>Min-Max Scaler</b> scales features to a given range, usually between 0 and 1 (or any other specified range). It rescales each feature by subtracting the minimum value and then dividing by the range, giving all features a common minimum and maximum.

<b>Robust Scaler</b> scales features using percentiles and median instead of mean and variance. It removes the median and scales the data based on the IQR (Interquartile Range), making it robust to outliers by using more robust estimates of central tendency.

In [ ]:
# No Scaling
users_without_scaling = users.copy()

# Standard Scaling
users_standard_scaling = StandardScaler().fit_transform(users)

# Min-Max Scaling
users_min_max_scaling = MinMaxScaler().fit_transform(users)

# Robust Scaling
users_robust_scaling = RobustScaler().fit_transform(users)

This section involves the application of clustering algorithms to derive the final solution. It's important to mention that all the methods outlined below were tested using the four datasets.

## <font color='#FFFFF'>2.3. Clustering <font><a class="anchor" id="p2_3"></a>
  
[Back to TOC](#toc)

<b>K Means</b>

The k-means algorithm partitions data into k clusters, each represented by a centroid. It aims to minimize intra-group variance by iteratively assigning data points to the nearest centroid and updating the centroids until convergence. The quality of clustering is based on the proximity of points within the same cluster. Adjusting the number of clusters (k) can impact the clustering's effectiveness in reducing the sum of squared errors.